# Lilylet NotaGen — autoregressive decoding test

Loads a trained `LilyletNotaGen` checkpoint and runs hierarchical (patch/char) autoregressive generation via `starry.lilylet.patchyGenerator.LilyletPatchyGenerator`.

The generator (adapted from NotaGen's `inference/inference.py`) encapsulates the whole loop:
1. Patch-level decoder encodes the patch sequence so far -> last hidden state.
2. Char-level decoder autoregressively samples the `patch_size` token ids inside the next patch, seeded by that hidden state.
3. The new patch is appended to the context; loop until an EOS patch `[bos, eos, ...]` appears or `max_patches` is hit.

Token ids are decoded back to text via the tokenizer's `text_by_id` table (NotaGen used raw `chr()`; Lilylet has protected multi-char tokens and ids > 127).

`generate()` also supports `measures=` (force the body to start at `[r:0/<measures>]`) and `postprocess=True` (drop `[r:x/y]` markers into trailing `% r:x/y` comments and insert blank lines).

In [13]:
from pathlib import Path
import sys
import os

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from starry.utils.config import Configuration
from starry.lilylet.patchyGenerator import LilyletPatchyGenerator

# A fully-trained checkpoint (lr0.2 run, epoch 999). Checkpoint sync is stopped;
# this best.chkpt is currently stable and usable.
CKPT = os.path.expanduser('~/data/models/deep-starry-logs/lilylet/20260606-lilylet-notagenx-large-lr0.2/best.chkpt')
CONFIG = str(REPO_ROOT / 'configs' / 'lilylet-notagenx-large.yaml')
# config stores a repo-relative tokenizer path; resolve it absolutely since the
# notebook kernel may run from tests/.
TOKENIZER = str(REPO_ROOT / 'assets' / 'lilylet-tokenizer.json')
print('repo:', REPO_ROOT)
print('checkpoint:', CKPT)

repo: /home/camus/work/deep-starry
checkpoint: /home/camus/data/models/deep-starry-logs/lilylet/20260606-lilylet-notagenx-large-lr0.2/best.chkpt


In [14]:
# Build the generator from the training config + checkpoint. from_config reads
# model.args (and the tokenizer path); we pass an absolute tokenizer path so it
# resolves regardless of the kernel's working directory.
config = Configuration.createOrLoad(CONFIG, volatile=True)
gen = LilyletPatchyGenerator.from_config(config, CKPT, tokenizer_path=TOKENIZER)

n_params = sum(p.numel() for p in gen.model.parameters())
print('params: %.2fM' % (n_params / 1e6))
print('device:', gen.device, '| patch_size:', gen.patch_size)
print('pad/bos/eos:', gen.pad_id, gen.bos_id, gen.eos_id)

params: 518.52M
device: cuda | patch_size: 16
pad/bos/eos: 0 1 2


In [15]:
# Sanity: encode a protected token + plain chars and decode it back via the generator.
demo = gen.tokenizer.encode('[r:0/3]')
print('encode "[r:0/3]" ->', demo)
print('decode back        ->', repr(gen.patch_to_text(demo)))

encode "[r:0/3]" -> [91, 114, 58, 48, 47, 51, 93]
decode back        -> '[r:0/3]'


In [16]:
# Unconditional generation (start from BOS only).
torch.manual_seed(0)
text = gen.generate(prompt_text='', max_patches=128, temperature=1.0, top_k=0, top_p=0.9, verbose=True)
print('\n\n===== generated text length:', len(text), 'chars =====')

[composer "Schumann, Robert"]
[genre "Romantic"]
[instrument "Keyboard"]
[r:0/45]\staff "1" \key e \major \time 3/4 \clef "treble" \tempo 4=144 r2. \\
\staff "2" \clef "bass" e,,8( b' e g b e |
[r:1/44]\staff "1" \key e \major \time 3/4 g''8( e b4 b' \\
\staff "2" r2. |
[r:2/43]\staff "1" \key e \major \time 3/4 \grace g''8 e'4.( d8 c4 \\
\staff "2" \clef "treble" e,8( b' e g b e |
[r:3/42]\staff "1" \key e \major \time 3/4 \grace c''8 g'4.( f8 e4 \\
\staff "2" g'8( b e g b r |
[r:4/41]\staff "1" \key e \major \time 3/4 \grace e''8 d'4.( c8 b4 \\
\staff "2" e,8( b' e g b e |
[r:5/40]\staff "1" \key e \major \time 3/4 \grace g'''8 f'4.( e8 c4 \\
\staff "2" g'8( b e g b r |
[r:6/39]\staff "1" \key e \major \time 3/4 \grace c'''8 g'4.( f8 e4 \\
\staff "2" e8( b' e g b e |
[r:7/38]\staff "1" \key e \major \time 3/4 \grace e'''8 d4.( c8 b4 \\
\staff "2" e8( b' e g b r |
[r:8/37]\staff "1" \key e \major \time 3/4 \grace b'''8 a'4.( g8 f4 \\
\staff "2" e8( b' e g b e |
[r:9/36]\staff "1" \key

In [17]:
# Conditional generation: seed with a metadata header and let the model continue.
torch.manual_seed(1)
prompt = '[title "Test Piece"]\n[composer "AI"]\n'
print('--- prompt ---')
print(prompt)
print('--- continuation ---')
text2 = gen.generate(prompt_text=prompt, max_patches=128, temperature=0.9, top_k=20, top_p=0.95, verbose=True)
print('\n\n===== total length:', len(text2), 'chars =====')

--- prompt ---
[title "Test Piece"]
[composer "AI"]

--- continuation ---
[title "Test Piece"]
[composer "AI"]
[genre "Classical"]
[instrument "Chamber"]
[r:127/0]\key d \major \time 3/4 \time 2/4 \tempo 4=16 \ottava #1 <d'' b'>8 \ottava #0 r r4 \bar "|." \\\
\time 1/4 <b'' d>8\ff r r4 \bar "|." \\\
\time 1/4 <b' d>8\ff r r4 \bar "|." \\\
\time 1/4 <b,, f' d' b'>8\ff r r4 \bar "|." |

[EOS patch -> stop]


===== total length: 313 chars =====


In [18]:
# Greedy (deterministic) decode for a reproducible smoke check.
torch.manual_seed(0)
text3 = gen.generate(prompt_text='', max_patches=64, temperature=1e-6, top_k=1, top_p=1.0, verbose=False)
print('greedy sample (first 400 chars):')
print(text3[:400])

greedy sample (first 400 chars):
[composer "Beethoven, Ludwig van"]
[genre "Classical"]
[instrument "Art Song"]
[r:0/45]\key a \major \time 6/8 \clef "treble" \tempo 4=92 r8 \\\
\staff "1" \clef "treble" e8\p \\
\staff "2" \clef "bass" r8 |
[r:1/44]\staff "1" \key a \major \time 6/8 r2. \\\
\staff "1" e8( a c e)( c a \\
\staff "2" <a c>4. <a c> |
[r:2/43]\staff "1" \key a \major \time 6/8 r2. \\\
\staff "1" a'8( g g)-. g( a b \\



In [19]:
# Showcase the extras: force the body to 8 measures and post-process the output
# (move [r:x/y] markers to trailing `% r:x/y` comments, insert blank lines).
torch.manual_seed(0)
clean = gen.generate(prompt_text='', max_patches=1024, temperature=0.9, top_k=20, top_p=0.95,
                     measures=8, postprocess=True, verbose=False)
print(clean)

[composer "Smetana, Bedrich"]
[genre "Romantic"]
[instrument "Keyboard"]

\staff "1" \key c \major \time 4/4 \clef "treble" \tempo 4=70 r1 \\
\staff "1" r8 <e a e'>4 <e a e'> <e a e'> <e a e'>8 \\
\staff "2" \clef "bass" r4 e,( a4. b8 | % r:0/8

\staff "1" \key c \major \time 4/4 r4 c'\p( b c \\
\staff "1" r8 <e a e'>4 <e a e'> <e a e'> <e a e'>8 \\
\staff "2" c4( a e e, | % r:1/7

\staff "1" \key c \major \time 4/4 d'4.( c8 b4 e \\
\staff "1" r8 <e gs e'>4 <e g e'> <e g e'> <e g e'>8 \\
\staff "2" r4 a( b4. e,8 | % r:2/6

\staff "1" \key c \major \time 4/4 c'4( b a a \\
\staff "1" r8 <e a e'>4 <e a e'> <e a e'> <e a e'>8 \\
\staff "2" c4( a g e | % r:3/5

\staff "1" \key c \major \time 4/4 b'4.( c8 d4 e \\
\staff "1" r8 <e gs e'>4 <e g e'> <e g e'> <e g e'>8 \\
\staff "2" r4 b( e4. b8 | % r:4/4

\staff "1" \key c \major \time 4/4 c'4( b a a \\
\staff "1" r8 <e a e'>4 <e a e'> <e a e'> <e a e'>8 \\
\staff "2" c4( a g e | % r:5/3

\staff "1" \key c \major \time 4/4 b'4.( c8 d4 e \\
\sta